# `nn` Package — User Guide & Documentation
A **from-scratch deep learning framework** built entirely on NumPy, designed for educational transparency.
This notebook serves as both a **tutorial** and a **reference** for all public APIs.

## Table of Contents
1. [Installation & Imports](#1-installation--imports)
2. [Core Concepts: Tensor & Autograd](#2-core-concepts-tensor--autograd)
3. [Layers Reference](#3-layers-reference)
4. [Model API (Sequential)](#4-model-api-sequential)
5. [Training Workflow](#5-training-workflow)
6. [Example A — CNN on Real Data](#6-example-a--cnn-digit-classification)
7. [Example B — SimpleRNN on Real Data](#7-example-b--simplernn-digit-classification)
8. [Example C — LSTM on Real Data](#8-example-c--lstm-digit-classification)
9. [Save & Load Weights](#9-save--load-weights)
10. [Advanced Features](#10-advanced-features)
11. [API Quick Reference](#11-api-quick-reference)

---
## 1. Installation & Imports
The package lives in `src/nn/`. Just make sure `src/` is on your Python path.

**Dependencies:** `numpy`, `scipy`, `tqdm` (optional, for progress bars), `sklearn` (for demo datasets only).

In [ ]:
import sys, os
import numpy as np
sys.path.insert(0, os.path.abspath('.'))

# Core imports
from nn import (
    Tensor, no_grad, Model,
    Dense, Conv2D, LocallyConnected2D,
    Flatten, MaxPooling2D, AveragePooling2D,
    GlobalMaxPooling2D, GlobalAveragePooling2D,
    Embedding, SimpleRNN, LSTM, RMSNorm,
    Adam, SGD, Input
)

print('All imports successful!')

---
## 2. Core Concepts: Tensor & Autograd
`Tensor` is the fundamental building block. It wraps a NumPy array and tracks operations
for automatic differentiation (backpropagation).

### Key Properties
| Property | Description |
|---|---|
| `.data` | The underlying `np.ndarray` |
| `.grad` | Gradient accumulated during `.backward()` |
| `.requires_grad` | If `True`, this tensor participates in gradient computation |

### Supported Operations
`+`, `-`, `*`, `/`, `**`, `@` (matmul), `sum()`, `mean()`, `reshape()`, `split()`, `concatenate()`, indexing/slicing — all with full autograd support.

In [ ]:
# Creating tensors
a = Tensor([1.0, 2.0, 3.0], requires_grad=True)
b = Tensor([4.0, 5.0, 6.0], requires_grad=True)

# Forward pass
c = (a * b).sum()  # dot product
print(f'c = {c.data}')

# Backward pass (autograd)
c.backward()
print(f'dc/da = {a.grad}  (should be [4, 5, 6])')
print(f'dc/db = {b.grad}  (should be [1, 2, 3])')

### `no_grad()` Context Manager
Use `no_grad()` to disable gradient tracking (e.g., during inference). This saves memory and speeds up computation.

In [ ]:
with no_grad():
    x = Tensor([1.0, 2.0], requires_grad=True)
    y = x * 2
    print(f'Gradient tracking disabled: y has no graph -> {y._prev}')

---
## 3. Layers Reference
All layers follow a consistent API: `layer.build(input_shape)` then `layer.forward(inputs)`.
When used inside a `Model`, building is **automatic**.

### Available Layers

| Layer | Constructor | Input Shape | Output Shape |
|---|---|---|---|
| `Input` | `Input(shape=(batch, ...))` | — | passthrough |
| `Dense` | `Dense(units, activation)` | `(B, features)` | `(B, units)` |
| `Conv2D` | `Conv2D(filters, kernel_size, stride, padding, activation)` | `(B, H, W, C)` | `(B, H', W', filters)` |
| `LocallyConnected2D` | Same as Conv2D but unshared weights | `(B, H, W, C)` | `(B, H', W', filters)` |
| `MaxPooling2D` | `MaxPooling2D(pool_size)` | `(B, H, W, C)` | `(B, H//p, W//p, C)` |
| `AveragePooling2D` | `AveragePooling2D(pool_size)` | `(B, H, W, C)` | `(B, H//p, W//p, C)` |
| `GlobalMaxPooling2D` | `GlobalMaxPooling2D()` | `(B, H, W, C)` | `(B, C)` |
| `GlobalAveragePooling2D` | `GlobalAveragePooling2D()` | `(B, H, W, C)` | `(B, C)` |
| `Flatten` | `Flatten()` | `(B, ...)` | `(B, prod(...))` |
| `Embedding` | `Embedding(vocab_size, embed_dim)` | `(B, seq_len)` | `(B, seq_len, embed_dim)` |
| `SimpleRNN` | `SimpleRNN(units, return_sequences)` | `(B, T, features)` | `(B, units)` or `(B, T, units)` |
| `LSTM` | `LSTM(units, return_sequences)` | `(B, T, features)` | `(B, units)` or `(B, T, units)` |
| `RMSNorm` | `RMSNorm()` | any | same |

### Available Activations
`'linear'`, `'relu'`, `'sigmoid'`, `'tanh'`, `'softmax'`, `'swish'`, `'gelu'`

### Available Initializers
`'zero'`, `'uniform'`, `'normal'`, `'xavier'`, `'he'`

---
## 4. Model API (Sequential)
The `Model` class provides a Keras-like sequential API.

### Two Ways to Build a Model
**Option A: Pass layers to constructor (recommended)**
```python
model = Model([
    Input(shape=(None, 784)),
    Dense(128, activation='relu'),
    Dense(10, activation='softmax')
], seed=42)
```

**Option B: Add layers incrementally**
```python
model = Model()
model.add(Dense(128, activation='relu'))
model.add(Dense(10, activation='softmax'))
# Layers are built lazily on first .fit() call
```

> **Note:** When using Option A with an `Input` layer, the model is built immediately.
> Without `Input`, building is deferred until `.fit()` or `.build()` is called.

In [ ]:
# Quick demo: build and summarize
demo = Model([
    Input(shape=(None, 4)),
    Dense(8, activation='relu'),
    Dense(3, activation='softmax')
], seed=42)

demo.summary()

---
## 5. Training Workflow

### Step 1: Compile
```python
model.compile(optimizer, loss, learning_rate=0.001)
```

| Optimizers | Losses |
|---|---|
| `'adam'`, `'sgd'` | `'mse'` — Mean Squared Error |
| Or pass an instance: `Adam(lr=0.01)` | `'bce'` — Binary Cross-Entropy |
| | `'cce'` — Categorical Cross-Entropy (one-hot) |
| | `'scce'` — Sparse Categorical Cross-Entropy (integer labels) |

### Step 2: Fit
```python
history = model.fit(X, y, epochs=10, batch_size=32, verbose=2, validation_data=(X_val, y_val))
```

| `verbose` | Behavior |
|---|---|
| `0` | Silent — no output |
| `1` | One line per epoch |
| `2` | tqdm progress bar per epoch (default) |

### Step 3: Predict & Evaluate
```python
predictions = model.predict(X_test)           # returns np.ndarray
loss_value  = model.evaluate(X_test, y_test)  # returns scalar loss
```

In [ ]:
# Minimal training example: XOR
X_xor = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
y_xor = np.array([[0],[1],[1],[0]], dtype=float)

model_xor = Model([
    Input(shape=(None, 2)),
    Dense(8, activation='tanh'),
    Dense(1, activation='sigmoid')
], seed=42)

model_xor.compile(optimizer='adam', loss='bce', learning_rate=0.05)
history = model_xor.fit(X_xor, y_xor, epochs=300, batch_size=4, verbose=0)

preds = model_xor.predict(X_xor)
print(f'Predictions: {np.round(preds.flatten(), 3)}')
print(f'Expected:    [0, 1, 1, 0]')
print(f'Final loss:  {history["train_loss"][-1]:.4f}')

---
## 6. Example A — CNN Digit Classification
Using `sklearn.datasets.load_digits` (8x8 grayscale images, 10 classes).

**Architecture:** `Conv2D(8, 3)` -> `MaxPooling2D(2)` -> `Flatten` -> `Dense(32)` -> `Dense(10, softmax)`

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

digits = load_digits()
X_all = digits.data / 16.0   # normalize to [0, 1]
y_all = digits.target

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=50, train_size=200, random_state=42, stratify=y_all
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}, Classes: {np.unique(y_train)}')

In [ ]:
# Reshape for Conv2D: (N, H, W, C)
X_train_cnn = X_train.reshape(-1, 8, 8, 1)
X_test_cnn  = X_test.reshape(-1, 8, 8, 1)

model_cnn = Model([
    Input(shape=(None, 8, 8, 1)),
    Conv2D(8, kernel_size=3, padding=0, activation='relu'),
    MaxPooling2D(pool_size=2),
    Flatten(),
    Dense(32, activation='relu'),
    Dense(10, activation='softmax')
], seed=42)

model_cnn.summary()
model_cnn.compile(optimizer='adam', loss='scce', learning_rate=0.005)

history_cnn = model_cnn.fit(
    X_train_cnn, y_train,
    epochs=15, batch_size=32, verbose=2,
    validation_data=(X_test_cnn, y_test)
)

preds_cnn = np.argmax(model_cnn.predict(X_test_cnn), axis=-1)
acc_cnn = np.mean(preds_cnn == y_test)
print(f'\nCNN Test Accuracy: {acc_cnn:.2%}')

---
## 7. Example B — SimpleRNN Digit Classification
Same dataset, but reshaped as **sequences**: each 8x8 image becomes 8 timesteps of 8 features.

**Architecture:** `SimpleRNN(32)` -> `Dense(10, softmax)`

In [ ]:
# Reshape for RNN: (N, timesteps, features)
X_train_rnn = X_train.reshape(-1, 8, 8)
X_test_rnn  = X_test.reshape(-1, 8, 8)

model_rnn = Model([
    Input(shape=(None, 8, 8)),
    SimpleRNN(32),
    Dense(10, activation='softmax')
], seed=42)

model_rnn.summary()
model_rnn.compile(optimizer='adam', loss='scce', learning_rate=0.005)

history_rnn = model_rnn.fit(
    X_train_rnn, y_train,
    epochs=30, batch_size=32, verbose=2,
    validation_data=(X_test_rnn, y_test)
)

preds_rnn = np.argmax(model_rnn.predict(X_test_rnn), axis=-1)
acc_rnn = np.mean(preds_rnn == y_test)
print(f'\nSimpleRNN Test Accuracy: {acc_rnn:.2%}')

---
## 8. Example C — LSTM Digit Classification
Same data as SimpleRNN for direct comparison. LSTM adds gating mechanisms (forget, input, output gates).

**Architecture:** `LSTM(32)` -> `Dense(10, softmax)`

In [ ]:
model_lstm = Model([
    Input(shape=(None, 8, 8)),
    LSTM(32),
    Dense(10, activation='softmax')
], seed=42)

model_lstm.summary()
model_lstm.compile(optimizer='adam', loss='scce', learning_rate=0.005)

history_lstm = model_lstm.fit(
    X_train_rnn, y_train,
    epochs=30, batch_size=32, verbose=2,
    validation_data=(X_test_rnn, y_test)
)

preds_lstm = np.argmax(model_lstm.predict(X_test_rnn), axis=-1)
acc_lstm = np.mean(preds_lstm == y_test)
print(f'\nLSTM Test Accuracy: {acc_lstm:.2%}')

---
### Results Comparison

In [ ]:
print('=' * 50)
print(f'{"Model":<15} {"Test Accuracy":>15} {"Final Loss":>12}')
print('=' * 50)
print(f'{"CNN":<15} {acc_cnn:>14.2%} {history_cnn["train_loss"][-1]:>12.4f}')
print(f'{"SimpleRNN":<15} {acc_rnn:>14.2%} {history_rnn["train_loss"][-1]:>12.4f}')
print(f'{"LSTM":<15} {acc_lstm:>14.2%} {history_lstm["train_loss"][-1]:>12.4f}')
print('=' * 50)

---
## 9. Save & Load Weights
Weights are saved as `.npz` files (NumPy compressed archives).

In [ ]:
# Save trained weights
model_cnn.save('demo_cnn_weights')
print('Saved!')

# Load into a fresh model with identical architecture
model_cnn_loaded = Model([
    Input(shape=(None, 8, 8, 1)),
    Conv2D(8, kernel_size=3, padding=0, activation='relu'),
    MaxPooling2D(pool_size=2),
    Flatten(),
    Dense(32, activation='relu'),
    Dense(10, activation='softmax')
])
model_cnn_loaded.load('demo_cnn_weights.npz')

# Verify predictions match
preds_original = model_cnn.predict(X_test_cnn)
preds_loaded   = model_cnn_loaded.predict(X_test_cnn)
print(f'Predictions match: {np.allclose(preds_original, preds_loaded)}')

os.remove('demo_cnn_weights.npz')  # cleanup

---
## 10. Advanced Features
### Freeze / Unfreeze Layers
Freezing a layer excludes its parameters from gradient updates (useful for transfer learning).

In [ ]:
model_freeze = Model([
    Input(shape=(None, 2)),
    Dense(8, activation='relu'),
    Dense(1, activation='sigmoid')
], seed=42)

# Freeze the first Dense layer
model_freeze.layers[1].freeze()
trainable = len(model_freeze.layers[1].parameters())
total     = len(model_freeze.layers[1].tensors())
print(f'Layer 1 trainable params: {trainable}')
print(f'Layer 1 total tensors:    {total}')

# Unfreeze
model_freeze.layers[1].unfreeze()
trainable = len(model_freeze.layers[1].parameters())
print(f'After unfreeze:           {trainable} trainable')

### Regularization (L1 / L2)
Pass `l1_lambda` and/or `l2_lambda` to any layer constructor.

In [ ]:
model_reg = Model([
    Input(shape=(None, 2)),
    Dense(8, activation='relu', l1_lambda=0.01, l2_lambda=0.001),
    Dense(1, activation='sigmoid')
], seed=42)

model_reg.compile(optimizer='adam', loss='bce')
history = model_reg.fit(X_xor, y_xor, epochs=100, batch_size=4, verbose=0)
print(f'Final loss (with regularization): {history["train_loss"][-1]:.4f}')

### Custom Optimizer Instance
Instead of passing a string, pass a configured optimizer object.

In [ ]:
from nn import Adam, SGD

opt = Adam(learning_rate=0.01)
model_custom = Model([
    Input(shape=(None, 2)),
    Dense(4, activation='relu'),
    Dense(1, activation='sigmoid')
])
model_custom.compile(optimizer=opt, loss='bce')
history = model_custom.fit(X_xor, y_xor, epochs=200, batch_size=4, verbose=0)
print(f'Final loss: {history["train_loss"][-1]:.4f}')

---
## 11. API Quick Reference

### Model
| Method | Description |
|---|---|
| `Model(layers, seed)` | Create model, optionally with layers and RNG seed |
| `.add(layer)` | Append a layer |
| `.build(input_shape)` | Manually trigger weight initialization |
| `.compile(optimizer, loss, learning_rate)` | Set optimizer and loss function |
| `.fit(X, y, epochs, batch_size, verbose, validation_data)` | Train the model |
| `.predict(X, batch_size)` | Run forward pass, returns `np.ndarray` |
| `.evaluate(X, y, batch_size)` | Compute loss on data |
| `.summary()` | Print layer/parameter table |
| `.save(filepath)` | Save weights to `.npz` |
| `.load(filepath)` | Load weights from `.npz` |
| `.parameters()` | List of trainable `Tensor` objects |

### Tensor
| Method | Description |
|---|---|
| `Tensor(data, requires_grad)` | Create a tensor |
| `.backward()` | Backpropagate gradients |
| `.data` | Raw NumPy array |
| `.grad` | Accumulated gradient |
| `Tensor.concatenate(tensors, axis)` | Concatenate tensors |
| `no_grad()` | Context manager to disable autograd |